In [1]:
import parsers
from common.utils.rclone import copy, list_remote
from subprocess import call
import os
import os
import matlab.engine
import re
import shutil
from common.utils.time import unix_to_timestamps
from common.utils.ingest import storage_format_date
import numpy as np
import pandas as pd

In [2]:
myRuneParser = parsers.RuneParser(r'/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data/rcs07/')

In [3]:
myRuneParser.parse_rune_from_rcs_timestamps()

/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data-net-common/common/utils/rune.py:119: UserWarning: No Data for accel in time range: 165609195 - 165611788
  warnings.warn(warning_message)
/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data-net-common/common/utils/rune.py:119: UserWarning: No Data for heart rate in time range: 165609195 - 165611788
  warnings.warn(warning_message)
/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data-net-common/common/utils/rune.py:119: UserWarning: No Data for tremor in time range: 165609195 - 165611788
  warnings.warn(warning_message)
/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data-net-common/common/utils/rune.py:119: UserWarning: No Data for tremor severity in time range: 165609195 - 165611788
  warnings.warn(warning_message)
/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data-net-common/common/utils/rune.py:119: UserWarning: No Data for dyskinesia in time range: 165609195 - 165611788
  warnings.warn(warning_mes

In [ ]:
os.chdir(r'/Users/raphaelb/Documents/UW/Research/gridlab/optimal/')

In [ ]:
#copy(r'/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data-net-subject/source_data/temp/test.txt', 'secret_sauce:/dir4/')

In [ ]:
list_of_ucsf_server_sessions = np.asarray(['Session1569834591963', 'Session1588356642957', 'Session1654791220617'])

In [ ]:
#Get dates folder names that has been processed and uploaded to wasabi
current_dates = list_remote('rcs07/rcs_v2/')
current_dates = [int(i[0:-2]) for i in current_dates]

#Find the most recent date on wasabi
lastest_date_index = np.argmax(current_dates)
lastest_date = current_dates[lastest_date_index]


#Convert list of sessions (Unix Date -> Timestamp -> Date -> Date as integer)
session_dates = np.asarray([int(unix_to_timestamps(i[7:]).strftime("%Y%m%d")) for i in list_of_ucsf_server_sessions])

#Check for any dates later than the most recent downloaded data
new_sessions_mask = session_dates > lastest_date
new_session_names = list_of_ucsf_server_sessions[new_sessions_mask]

#Download the foldes in the new_session_names folder TODO: when we get ucsf server
print(new_session_names)



In [ ]:
list_remote('rcs07/')

In [ ]:
from common.utils.rune import get_client, read_fields,get_watch_data,get_bilateral_watch_data,get_watch_data,make_df_from_rune_accessor

In [ ]:
myclient = get_client()

In [ ]:
time_range = [1653594400,1653604400]

wrist_params = {
'patient_id': 'rcs07',
'left_watch_id': '8QuY9OFb',
'right_watch_id': 'RElEtNme',
'time_range': time_range
}
rcs_params = {
'patient_id': 'rcs07',
'left_watch_id': 'NPC700419H',
'right_watch_id': 'NPC700403H',
'time_range': time_range
}

In [ ]:
get_bilateral_watch_data(myclient, 'lfp', **rcs_params)

In [ ]:
get_bilateral_watch_data(myclient, 'band power', **rcs_params)

In [ ]:
get_bilateral_watch_data(myclient, 'accel', **wrist_params)

In [ ]:
get_bilateral_watch_data(myclient, 'heart rate', **wrist_params)

In [ ]:
get_bilateral_watch_data(myclient, 'tremor', **wrist_params)

In [ ]:
get_bilateral_watch_data(myclient, 'tremor severity', **wrist_params)

In [ ]:
get_bilateral_watch_data(myclient, 'dyskinesia', **wrist_params)

In [ ]:

my_accel = get_bilateral_watch_data(myclient, 'accel', **wrist_params)
my_rotation = get_bilateral_watch_data(myclient, 'rotation', **wrist_params)
my_heart_rate = (make_df_from_rune_accessor(myclient.HeartRate(**left_watch_params)),(make_df_from_rune_accessor(myclient.HeartRate(**right_watch_params))))
my_tremor = (make_df_from_rune_accessor(myclient.ProbabilitySymptom(symptom='tremor',**left_watch_params)),(make_df_from_rune_accessor(myclient.ProbabilitySymptom(symptom='tremor',**right_watch_params))))
my_tremor_severity = (make_df_from_rune_accessor(myclient.ProbabilitySymptom(symptom='tremor',severity='*',**left_watch_params)),(make_df_from_rune_accessor(myclient.ProbabilitySymptom(symptom='tremor',severity='*',**right_watch_params))))
my_dyskinesia = (make_df_from_rune_accessor(myclient.ProbabilitySymptom(symptom='dyskinesia',**left_watch_params)),(make_df_from_rune_accessor(myclient.ProbabilitySymptom(symptom='dyskinesia',**right_watch_params))))




In [ ]:
make_df_from_rune_accessor(myclient.BandPower(**right_rcs_params))


In [ ]:
def get_timestamps(path_to_folder):
    neural_time_domain = pd.read_csv(path_to_folder + '/NeuralTimeDomain.csv')
    start = str(neural_time_domain["timestamp"].iloc[0])
    end = str(neural_time_domain["timestamp"].iloc[-1])
    time_range = [start[0:-4], end[0:-4]]
    return time_range

def get_side(folder_name):
    if 'left' in folder_name:
        return True
    if 'right' in folder_name:
        return False
    
def get_params(side,dual_sided_params):
    if side:
        return {
        'patient_id': dual_sided_params['patient_id'],
        'device_id': dual_sided_params['left_watch_id'],
        'start_time': dual_sided_params['time_range'][0],
        'end_time': dual_sided_params['time_range'][1]}
    else:
        return {
        'patient_id': dual_sided_params['patient_id'],
        'device_id': dual_sided_params['right_watch_id'],
        'start_time': dual_sided_params['time_range'][0],
        'end_time': dual_sided_params['time_range'][1]}
    
    
def timestamps_to_rune_data(timestamps,folder_name):


    wrist_params = {
    'patient_id': 'rcs07',
    'left_watch_id': '8QuY9OFb',
    'right_watch_id': 'RElEtNme',
    'time_range': timestamps
    }
    rcs_params = {
    'patient_id': 'rcs07',
    'left_watch_id': 'NPC700419H',
    'right_watch_id': 'NPC700403H',
    'time_range': timestamps
    }
        
    
    my_accel = get_watch_data(myclient, get_params(get_side(folder_name),wrist_params), 'accel').set_index('timestamp')
    my_rotation = get_watch_data(myclient, get_params(get_side(folder_name),wrist_params), 'rotation').set_index('timestamp')
    my_heart_rate = get_watch_data(myclient, get_params(get_side(folder_name),wrist_params), 'heart rate').set_index('timestamp')
    my_tremor = get_watch_data(myclient, get_params(get_side(folder_name),wrist_params), 'tremor').set_index('timestamp')
    my_tremor_severity = get_watch_data(myclient, get_params(get_side(folder_name),wrist_params), 'tremor severity').set_index('timestamp')
    my_dyskinesia = get_watch_data(myclient, get_params(get_side(folder_name),wrist_params), 'dyskinesia').set_index('timestamp')
    my_lfp = get_watch_data(myclient, get_params(get_side(folder_name),rcs_params), 'lfp').set_index('timestamp')
    my_band_power = get_watch_data(myclient, get_params(get_side(folder_name),rcs_params), 'band power').set_index('timestamp')
    
    return my_accel, my_rotation, my_heart_rate, my_tremor, my_tremor_severity, my_dyskinesia, my_lfp, my_band_power

def output_to_csv(path, date, folder, my_accel, my_rotation, my_heart_rate, my_tremor, my_tremor_severity, my_dyskinesia, my_lfp, my_band_power):

    if get_side(folder):
        full_path =  path + 'rune/' + date + '/rune_left_' + folder[-27:]
    else:
        full_path =  path + 'rune/' + date + '/rune_right_' + folder[-27:]
        
    # If folder doesn't exist, then create it.
    if not os.path.isdir(full_path):
        os.makedirs(full_path)

    
    my_accel.to_csv(full_path + '/accel.csv')
    my_rotation.to_csv(full_path + '/rotation.csv')
    my_heart_rate.to_csv(full_path + '/heart_rate.csv')
    my_tremor.to_csv(full_path + '/tremor.csv')
    my_tremor_severity.to_csv(full_path + '/tremor_severity.csv')
    my_dyskinesia.to_csv(full_path + '/dyskinesia.csv')
    my_lfp.to_csv(full_path + '/lfp.csv')
    my_band_power.to_csv(full_path + '/band_power.csv')


In [ ]:
folder_path = '/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data/rcs07/'
date_dir = os.listdir(folder_path + 'rcs/combined_anonymized_json_csv/')

if '.DS_Store' in date_dir:
    date_dir.remove('.DS_Store')

for i in date_dir:
    session_dir = os.listdir(folder_path + 'rcs/combined_anonymized_json_csv/'+i)
    if '.DS_Store' in session_dir:
        session_dir.remove('.DS_Store')
    for j in session_dir:
        my_timestamps = get_timestamps(folder_path + 'rcs/combined_anonymized_json_csv/'+i+'/'+j)    
        output_to_csv(folder_path,i,j,*timestamps_to_rune_data(my_timestamps,j))

## Dropbox Code

In [ ]:
#### DROPBOX_ACCESS_TOKEN = 'sl.BJ_794kQETs-AQ7jWJnc--vK2fCQ6RoPolVzvQ4vZEKdYfmYsr5WinUt3b-BCHlEEKjPuV59oGGaUA2ZPXI3JSyNy2F4MTe5bCVegs8z5WbXes0YZT7BF7H4tzX7K_HJPOrXo_Pc'
def dropbox_connect():
    """Create a connection to Dropbox."""

    try:
        dbx = dropbox.Dropbox(DROPBOX_ACCESS_TOKEN)
    except AuthError as e:
        print('Error connecting to Dropbox with access token: ' + str(e))
    return dbx

def dropbox_list_files():
    """Return a Pandas dataframe of files in a given Dropbox folder path in the Apps directory.
    """

    dbx = dropbox_connect()

    try:
        files = dbx.sharing_list_folders().entries
        files_list = []
        for file in files:
            if isinstance(file, dropbox.files.FileMetadata):
                metadata = {
                    'name': file.name,
                    'path_display': file.path_display,
                    'client_modified': file.client_modified,
                    'server_modified': file.server_modified
                }
                files_list.append(metadata)

        df = pd.DataFrame.from_records(files_list)
        return df.sort_values(by='server_modified', ascending=False)

    except Exception as e:
        print('Error getting list of files from Dropbox: ' + str(e))

dbx = dropbox.Dropbox(DROPBOX_ACCESS_TOKEN)
unscyned_shared_link = dbx.sharing_list_folders().entries[1].preview_url
dbx.files_list_folder('/SummitData/SummitContinuousBilateralStreaming/RCS07L',recursive=False, shared_link=dropbox.files.SharedLink(url=unscyned_shared_link)).entries[456]

In [ ]:
dbx.sharing_get_shared_link_file_to_file('/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data/rcs07/rcs/combined_original',unscyned_shared_link, path='/SummitData/SummitContinuousBilateralStreaming/RCS07L/Session1651792726456/.')

